In [1]:
import pandas as pd
from imperandi.ingest.clean import * # import clean as clean_module
from imperandi.utils.geometry import as_float_array


In [2]:
import logging
logging.basicConfig(level=logging.DEBUG)

In [3]:
!pwd

/mnt/Data/Code/IMPERANDI/tests/experiment


In [4]:
df = pd.read_csv("../data/dicom_index.csv")

# For testing, we will create a pseudo volume_id and assume all rows belong to the same patient, study, and series.
#df["volume_id"] = "pseudo_volume_id"  # Placeholder for testing
df["patient_key"] = "same_patient_id"  # Assuming patient_id is the same as patient_key for testing
df["study_id"] = "same_study_id"  # Assuming study_id is the same for testing
df["series_id"] = "same_series_id"  # Assuming series_id is the same

In [ ]:
preferred_cols = [
        "patient_key",
        "study_id",
        "series_id",
        "ImageType",
        "AcquisitionNumber",
        "ImageOrientationPatient",
        "SliceThickness",
        "PixelSpacingXY",
    ]

df[preferred_cols]

In [5]:
report_volumes(df, "initial load")

if "ImageOrientationPatient" in df.columns:
    df_prev = df.copy()
    df["ImageOrientationPatient"] = df["ImageOrientationPatient"].apply(
        standardize_iop
    )
    df = filter_by_acquisition_plane(df)
    report_volumes(df, "keeping only AXIAL acquisitions")
    report_change(df, df_prev)
    
df_prev = df.copy()
df = generate_volume_id(df)
report_volumes(df, "generating volume IDs")
report_change(df, df_prev)

INFO:imperandi.utils.misc:After initial load:
INFO:imperandi.utils.misc:	Unique patients: 1
INFO:imperandi.utils.misc:	Unique studies:  1
INFO:imperandi.utils.misc:	Unique series:   1
INFO:imperandi.utils.misc:After keeping only AXIAL acquisitions:
INFO:imperandi.utils.misc:	Unique patients: 1
INFO:imperandi.utils.misc:	Unique studies:  1
INFO:imperandi.utils.misc:	Unique series:   1
INFO:imperandi.ingest.clean:For unique volume ID generation, using columns: ['patient_key', 'study_id', 'series_id', 'ImageType', 'AcquisitionNumber', 'ImageOrientationPatient', 'SliceThickness']
INFO:imperandi.utils.misc:After generating volume IDs:
INFO:imperandi.utils.misc:	Unique patients: 1
INFO:imperandi.utils.misc:	Unique studies:  1
INFO:imperandi.utils.misc:	Unique series:   1
INFO:imperandi.utils.misc:	Unique volumes:  1


In [6]:
display(df[["ImageOrientationPatient", "ImagePositionPatient"]])

,ImageOrientationPatient,ImagePositionPatient
0,"(1.0, 0.0, 0.0, 0.0, 1.0, 0.0)","[0, 0, 0]"
1,"(1.0, 0.0, 0.0, 0.0, 1.0, 0.0)","[0, 0, 1.60000002384186]"
2,"(1.0, 0.0, 0.0, 0.0, 1.0, 0.0)","[0, 0, 16.0000002384186]"
3,"(1.0, 0.0, 0.0, 0.0, 1.0, 0.0)","[0, 0, 160.000002384186]"
4,"(1.0, 0.0, 0.0, 0.0, 1.0, 0.0)","[0, 0, 161.600002408028]"
...,...,...
296,"(1.0, 0.0, 0.0, 0.0, 1.0, 0.0)","[0, 0, 152.000002264977]"
297,"(1.0, 0.0, 0.0, 0.0, 1.0, 0.0)","[0, 0, 153.600002288818]"
298,"(1.0, 0.0, 0.0, 0.0, 1.0, 0.0)","[0, 0, 155.20000231266]"
299,"(1.0, 0.0, 0.0, 0.0, 1.0, 0.0)","[0, 0, 156.800002336502]"


In [7]:
df = correct_volume_ids(df)
report_volumes(df, "correcting volume IDs")

display(df)

INFO:imperandi.utils.misc:After correcting volume IDs:
INFO:imperandi.utils.misc:	Unique patients: 1
INFO:imperandi.utils.misc:	Unique studies:  1
INFO:imperandi.utils.misc:	Unique series:   1
INFO:imperandi.utils.misc:	Unique volumes:  1


,dicom_path,PatientID,PatientName,StudyInstanceUID,SeriesInstanceUID,SOPInstanceUID,Modality,ModalitiesInStudy,SOPClassUID,Manufacturer,...,BurnedInAnnotation,dicom_filename,patient_key,study_id,series_id,_patient_key_raw,acquisition_plane,acquisition_angle,acquisition_axis,volume_id
0,tests/data/IRCAD_DICOM/3Dircadb1.1/exam/image_0,NaN,liver_01^patient,1.2.826.0.1.3680043.2.1125.6907248010745639067...,1.2.826.0.1.3680043.2.1125.9647753419956608505...,1.2.826.0.1.3680043.2.1125.3117066890503337486...,CT,NaN,1.2.840.10008.5.1.4.1.1.2,NaN,...,NaN,image_0,same_patient_id,same_study_id,same_series_id,liver_01^patient,AX,0.0,Z,0175940c80893fb6d89144207389a39c7dd7078a
1,tests/data/IRCAD_DICOM/3Dircadb1.1/exam/image_1,NaN,liver_01^patient,1.2.826.0.1.3680043.2.1125.6907248010745639067...,1.2.826.0.1.3680043.2.1125.9647753419956608505...,1.2.826.0.1.3680043.2.1125.9992007335188132999...,CT,NaN,1.2.840.10008.5.1.4.1.1.2,NaN,...,NaN,image_1,same_patient_id,same_study_id,same_series_id,liver_01^patient,AX,0.0,Z,0175940c80893fb6d89144207389a39c7dd7078a
2,tests/data/IRCAD_DICOM/3Dircadb1.1/exam/image_10,NaN,liver_01^patient,1.2.826.0.1.3680043.2.1125.6907248010745639067...,1.2.826.0.1.3680043.2.1125.9647753419956608505...,1.2.826.0.1.3680043.2.1125.4276323034127557047...,CT,NaN,1.2.840.10008.5.1.4.1.1.2,NaN,...,NaN,image_10,same_patient_id,same_study_id,same_series_id,liver_01^patient,AX,0.0,Z,0175940c80893fb6d89144207389a39c7dd7078a
3,tests/data/IRCAD_DICOM/3Dircadb1.1/exam/image_100,NaN,liver_01^patient,1.2.826.0.1.3680043.2.1125.6907248010745639067...,1.2.826.0.1.3680043.2.1125.9647753419956608505...,1.2.826.0.1.3680043.2.1125.7776439881718010678...,CT,NaN,1.2.840.10008.5.1.4.1.1.2,NaN,...,NaN,image_100,same_patient_id,same_study_id,same_series_id,liver_01^patient,AX,0.0,Z,0175940c80893fb6d89144207389a39c7dd7078a
4,tests/data/IRCAD_DICOM/3Dircadb1.1/exam/image_101,NaN,liver_01^patient,1.2.826.0.1.3680043.2.1125.6907248010745639067...,1.2.826.0.1.3680043.2.1125.9647753419956608505...,1.2.826.0.1.3680043.2.1125.3420595866513858350...,CT,NaN,1.2.840.10008.5.1.4.1.1.2,NaN,...,NaN,image_101,same_patient_id,same_study_id,same_series_id,liver_01^patient,AX,0.0,Z,0175940c80893fb6d89144207389a39c7dd7078a
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
296,tests/data/IRCAD_DICOM/3Dircadb1.2/exam/image_95,NaN,liver_02^patient,1.2.826.0.1.3680043.2.1125.5998964760887913365...,1.2.826.0.1.3680043.2.1125.3253956498577534545...,1.2.826.0.1.3680043.2.1125.8762093271534024023...,CT,NaN,1.2.840.10008.5.1.4.1.1.2,NaN,...,NaN,image_95,same_patient_id,same_study_id,same_series_id,liver_02^patient,AX,0.0,Z,0175940c80893fb6d89144207389a39c7dd7078a
297,tests/data/IRCAD_DICOM/3Dircadb1.2/exam/image_96,NaN,liver_02^patient,1.2.826.0.1.3680043.2.1125.5998964760887913365...,1.2.826.0.1.3680043.2.1125.3253956498577534545...,1.2.826.0.1.3680043.2.1125.1858974063937818252...,CT,NaN,1.2.840.10008.5.1.4.1.1.2,NaN,...,NaN,image_96,same_patient_id,same_study_id,same_series_id,liver_02^patient,AX,0.0,Z,0175940c80893fb6d89144207389a39c7dd7078a
298,tests/data/IRCAD_DICOM/3Dircadb1.2/exam/image_97,NaN,liver_02^patient,1.2.826.0.1.3680043.2.1125.5998964760887913365...,1.2.826.0.1.3680043.2.1125.3253956498577534545...,1.2.826.0.1.3680043.2.1125.1423267725229952141...,CT,NaN,1.2.840.10008.5.1.4.1.1.2,NaN,...,NaN,image_97,same_patient_id,same_study_id,same_series_id,liver_02^patient,AX,0.0,Z,0175940c80893fb6d89144207389a39c7dd7078a
299,tests/data/IRCAD_DICOM/3Dircadb1.2/exam/image_98,NaN,liver_02^patient,1.2.826.0.1.3680043.2.1125.5998964760887913365...,1.2.826.0.1.3680043.2.1125.3253956498577534545...,1.2.826.0.1.3680043.2.1125.2135664643368385871...,CT,NaN,1.2.840.10008.5.1.4.1.1.2,NaN,...,NaN,image_98,same_patient_id,same_study_id,same_series_id,liver_02^patient,AX,0.0,Z,0175940c80893fb6d89144207389a39c7dd7078a
